# Ligature: training runs on Kaggle or Colab

Runs the Phase 4 experiments and packs everything needed locally into one zip.

- Kaggle: Accelerator "GPU T4 x2" and Internet on. Colab: any one-GPU runtime.
- Every cell waits until its work has finished, so "Save & Run All" in the background works too.
- The code is cloned from GitHub, so push your latest commits before running.
- With two GPUs each pair trains side by side; with one GPU the runs go one after another.

In [ ]:
MAX_STEPS = None  # e.g. 200 for a quick check; passed to every run so token budgets stay equal
RUN_SECOND_SEED = True  # pair 3: base and base_learned again with --seed 2
REPO_URL = "https://github.com/kshubham-108/ligature"
POLL_SECONDS = 60  # how often to print each run's latest progress line

## Environment

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE:
    # /kaggle/temp is not saved as output, which keeps the 2 GB of raw text out of it.
    REPO, OUT_DIR = Path("/kaggle/temp/ligature"), Path("/kaggle/working")
elif ON_COLAB:
    REPO, OUT_DIR = Path("/content/ligature"), Path("/content")
else:
    raise RuntimeError("This notebook expects to run on Kaggle or Colab.")

N_GPUS = torch.cuda.device_count()
if N_GPUS == 0:
    raise RuntimeError("No GPU visible. Kaggle: Accelerator > GPU T4 x2. Colab: a GPU runtime.")

# PYTHONPATH=src stands in for pip install -e .; unbuffered output keeps the run logs current.
ENV = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join(p for p in (str(REPO / "src"), os.environ.get("PYTHONPATH")) if p),
    "PYTHONUNBUFFERED": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
}


def run(cmd, cwd=None):
    """Run a command, stream its output into this cell, and raise if it fails."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
    except BaseException:
        proc.kill()  # stopping the cell must not leave the command running
        raise
    if proc.returncode != 0:
        raise RuntimeError(f"exit code {proc.returncode}: {' '.join(map(str, cmd))}")


print("Python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA", torch.version.cuda, f"| {N_GPUS} GPU(s)")
run(["nvidia-smi"])

## Code and dependencies

In [ ]:
if (REPO / ".git").exists():
    print(f"{REPO} is already cloned; skipping")
else:
    REPO.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
run(["git", "log", "-1", "--oneline"], cwd=REPO)

In [ ]:
# Import name -> pip name. torch is never installed here: the platform build matches its CUDA.
REQUIRED = {
    "regex": "regex",
    "yaml": "pyyaml",
    "huggingface_hub": "huggingface_hub",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
}
missing = [pip for module, pip in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All dependencies are already installed.")

## Data and bigram baseline

In [ ]:
if (REPO / "data" / "train.bin").exists() and (REPO / "data" / "val.bin").exists():
    print("data/train.bin and data/val.bin exist; skipping prepare_data.py")
else:
    run([sys.executable, "scripts/prepare_data.py"], cwd=REPO)
print((REPO / "results" / "data_stats.json").read_text())

In [ ]:
run([sys.executable, "scripts/bigram_baseline.py"], cwd=REPO)
print((REPO / "results" / "bigram.json").read_text())

## Training

`launch_runs` takes a list of `(config, run name, GPU, extra args)`. Each run's output goes to
`runs/<name>/train.log`. It prints the latest progress line of every run each `POLL_SECONDS`,
returns once all runs have exited, and raises if any of them failed.

In [ ]:
def latest_progress(log_path):
    """Return the last 'step ...' line of a run log, or a short placeholder."""
    if not log_path.exists():
        return "(no log yet)"
    lines = log_path.read_text(errors="replace").splitlines()
    steps = [line for line in lines if line.startswith("step ")]
    return steps[-1] if steps else (lines[-1] if lines else "(no output yet)")


def run_together(batch):
    """Start every run in the batch at once and wait for all of them."""
    procs = {}
    try:
        for config, name, gpu, extra in batch:
            run_dir = REPO / "runs" / name
            run_dir.mkdir(parents=True, exist_ok=True)
            cmd = [sys.executable, "-m", "ligature.train", "--config", config, "--out", f"runs/{name}"]
            cmd += [str(arg) for arg in extra]
            if MAX_STEPS is not None:
                cmd += ["--max_steps", str(MAX_STEPS)]
            log = open(run_dir / "train.log", "w")
            env = {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
            proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
            procs[name] = (proc, log, run_dir / "train.log")
            print(f"started {name} on GPU {gpu}: {' '.join(cmd[1:])}", flush=True)

        last_report = time.monotonic()
        while any(proc.poll() is None for proc, _, _ in procs.values()):
            time.sleep(5)
            if time.monotonic() - last_report >= POLL_SECONDS:
                for name, (_, _, log_path) in procs.items():
                    print(f"{name:>16} | {latest_progress(log_path)}", flush=True)
                last_report = time.monotonic()
    except BaseException:
        for proc, _, _ in procs.values():
            proc.kill()  # stopping the cell must not leave training running on the GPUs
        raise
    finally:
        for _, log, _ in procs.values():
            log.close()

    failed = []
    for name, (proc, _, log_path) in procs.items():
        print(f"{name} exited with code {proc.returncode} | {latest_progress(log_path)}")
        if proc.returncode != 0:
            failed.append(name)
            tail = log_path.read_text(errors="replace").splitlines()[-20:]
            print(f"--- last lines of {log_path} ---", *tail, sep="\n")
    if failed:
        raise RuntimeError(f"runs failed: {failed}")


def launch_runs(runs):
    """Train runs side by side, one per GPU; with a single GPU, one after another on GPU 0."""
    if N_GPUS > 1:
        run_together(runs)
    else:
        for config, name, _, extra in runs:
            run_together([(config, name, 0, extra)])

In [ ]:
PAIR_1 = [
    ("configs/base.yaml", "base", 0, []),
    ("configs/base_learned.yaml", "base_learned", 1, []),
]
launch_runs(PAIR_1)

In [ ]:
PAIR_2 = [
    ("configs/size_small.yaml", "size_small", 0, []),
    ("configs/size_medium.yaml", "size_medium", 1, []),
]
launch_runs(PAIR_2)

In [ ]:
PAIR_3 = [
    ("configs/base.yaml", "base_s2", 0, ["--seed", 2]),
    ("configs/base_learned.yaml", "base_learned_s2", 1, ["--seed", 2]),
]
if RUN_SECOND_SEED:
    launch_runs(PAIR_3)
else:
    print("RUN_SECOND_SEED is False; skipping pair 3")

## Results zip

Extracting `ligature_results.zip` at the repo root gives `results/data_stats.json`,
`results/bigram.json`, `results/runs/<name>/` (log.csv, config.yaml, run_info.yaml) and
`runs/base/` (ckpt.pt, tokenizer.json).

In [ ]:
def build_results_zip(repo, zip_path, run_names):
    """Zip the result files with paths relative to the repo root; raise if any are missing."""
    entries = {
        "results/data_stats.json": repo / "results" / "data_stats.json",
        "results/bigram.json": repo / "results" / "bigram.json",
    }
    for name in run_names:
        for filename in ("log.csv", "config.yaml", "run_info.yaml"):
            entries[f"results/runs/{name}/{filename}"] = repo / "runs" / name / filename
    for filename in ("ckpt.pt", "tokenizer.json"):
        entries[f"runs/base/{filename}"] = repo / "runs" / "base" / filename

    missing = [str(path) for path in entries.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"missing result files: {missing}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for arcname, path in entries.items():
            zf.write(path, arcname)
    print(*entries, sep="\n")
    print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")


run_names = [name for _, name, _, _ in PAIR_1 + PAIR_2 + (PAIR_3 if RUN_SECOND_SEED else [])]
build_results_zip(REPO, OUT_DIR / "ligature_results.zip", run_names)